In [ ]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf
from scipy.stats import pearsonr

from rich.progress import track

import seaborn as sns

## Download the data

In [ ]:
generator = nl.data.MNISTGenerator(ds_size=2000)

In [ ]:
network = stax.serial(
        stax.Flatten(),
        stax.Dense(128),
        stax.Relu(),
        stax.Dense(128),
        stax.Relu(),
        stax.Dense(10),
    )

optimizer=nl.optimizers.TraceOptimizer(scale_factor=1.0)
model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                input_shape=(1, 28, 28, 1),
                batch_size=5
        )

## Eigenvalue distributions

In [ ]:
ds_sizes = [5, 10, 20, 100, 200, 300, 400, 500, 700, 1000]
ds_ensembles = 5
model_ensembles = 5
eig_data = {item: [] for item in ds_sizes}
trace_data = {item: [] for item in ds_sizes}

for _ in track(range(model_ensembles)):
    model = nl.models.NTModel(
                nt_module=network,
                optimizer=optimizer,
                input_shape=(1, 28, 28, 1),
                batch_size=5
        )
    for ds_size in ds_sizes:
        for _ in range(model_ensembles):
            indices = np.random.randint(
                0, generator.train_ds["inputs"].shape[0] - 1, size=ds_size
            )
            ds = np.take(generator.train_ds["inputs"], indices, axis=0)
            
            matrix = model.compute_ntk(ds)["empirical"]
            
            eigs, _ = np.linalg.eigh(matrix)
            trace = np.trace(matrix)
            
            eig_data[ds_size].append(eigs)
            trace_data[ds_size].append(trace)

In [ ]:
test = eig_data[500]

# for item in test[0]:
sns.histplot(x=np.mean(eig_data[500], axis=0) / 2000, bins=20, stat="probability", discrete=False, label=2000)
sns.histplot(x=np.mean(eig_data[100], axis=0) / 1000, bins=20, stat="probability", discrete=False, label=1000)
sns.histplot(x=np.mean(eig_data[20], axis=0) / 500, bins=20, stat="probability", discrete=False, label=500)
sns.histplot(x=np.mean(eig_data[10], axis=0) / 100, bins=20, stat="probability", discrete=False, label=100)

# sns.kdeplot(np.mean(eig_data[2000], axis=0), label=2000)
# sns.kdeplot(np.mean(eig_data[1000], axis=0), label=1000)
# sns.kdeplot(np.mean(eig_data[500], axis=0), label=500)
# sns.kdeplot(np.mean(eig_data[100], axis=0), label=100)


plt.legend()
# plt.xlim(-10, 50)
# plt.yscale("log")
plt.show()

In [ ]:

huge = np.mean([item / item.sum() for item in eig_data[500]], axis=0)
large = np.mean([item / item.sum() for item in eig_data[100]], axis=0)
medium = np.mean([item / item.sum() for item in eig_data[20]], axis=0)
small = np.mean([item / item.sum() for item in eig_data[10]], axis=0)


sns.kdeplot(huge, label=500, fill=True)
sns.kdeplot(large, label=100, fill=True)
sns.kdeplot(medium, label=20, fill=True)
sns.kdeplot(small, label=10, fill=True)


plt.legend()
# plt.xlim(-0.1, 0.1)
plt.show()

In [ ]:
for item in trace_data:
    plt.plot(item, np.mean(trace_data[item]) / item, '.')